# Generate Figures & Tables — Revision 2 (consolidated)

This notebook regenerates **every revision-2 figure, panel and table** by running the
reproducible standalone scripts under `scripts/revision2/` (the single source of truth)
and displaying each result inline. It replaces the scattered, partly outdated figure
cells of `generate_figures_and_tables_revised.ipynb` for the updated set.

> **Run with the `cardiokoop` conda environment / kernel** (needs torch + the frozen
> Koopman checkpoint).

**Manuscript mapping**

| Script | Produces |
|---|---|
| `build_tables_figures.py` | Tables 3/4/5, Figure 5 (b–f), Figure 6, Figure S9 (b–f) |
| `regenerate_panels_a_gj.py` | Figure 5 a + g–j, Figure S9 a + g–j |
| `task_b_gamma_sweep.py` | Figure S7 (control-gain γ sweep) |
| `task_c_activation.py` | Table S4 (control-net activation comparison) |
| `task_d_realistic_streaming_noise.py` | Figure S8 (realistic + streaming noise) |
| `task_e_mode_ablation_export.py` | mode-ablation table export |
| `task_f_clean_reanchoring.py` | Figure S10 (clean re-anchoring sweep) |
| `figure_mode_analysis.py` | Figure S6 (Koopman mode analysis) |
| `figure_latent_hparam.py` | Figure S4 (latent / hyperparameter ablation) |
| `task_a_dlinear_nlinear.py` | trains the DLinear/NLinear baselines (**run once**) |

**Original** Figures 2–4 and Supplementary S1–S3/S5 are unchanged in this revision and
remain in `generate_figures_and_tables_revised.ipynb` (see the last section).


In [ ]:
import os, sys, subprocess, time
from pathlib import Path
from IPython.display import Image, display

# locate the repo root (the folder that contains scripts/revision2)
_here = Path.cwd()
REPO = next((p for p in [_here, *_here.parents]
             if (p / "scripts" / "revision2" / "run_all.py").exists()), None)
assert REPO is not None, "Could not locate the repo root (scripts/revision2/run_all.py)"
REV2   = REPO / "scripts" / "revision2"
FIGDIR = REPO / "results" / "figures"
PY     = sys.executable
print("Repo   :", REPO)
print("Python :", PY)

def run_script(name):
    """Run a revision-2 script in a fresh process (Agg backend) and stream its output."""
    script = REV2 / name
    print(f"$ python {script.relative_to(REPO)}\n")
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    r = subprocess.run([PY, str(script)], cwd=str(REPO), env=env,
                       capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print("STDERR:\n", r.stderr)
        raise RuntimeError(f"{name} exited with code {r.returncode}")

def show(*pngs, width=1100):
    for p in pngs:
        fp = FIGDIR / p
        display(Image(filename=str(fp), width=width)) if fp.exists() else print("MISSING:", fp)


## 0. (Optional) Train the direct linear baselines — Task A

Task A trains DLinear / NLinear (expensive; ~thousands of epochs on a tiny linear model).
It only needs to run **once** — its checkpoints/pickles are consumed by
`build_tables_figures.py`. The cell below runs it **only if** the pickles are missing.


In [ ]:
need_a = not ((REPO/"results"/"dlinear"/"dlinear_postprocessing_results.pkl").exists()
               and (REPO/"results"/"nlinear"/"nlinear_postprocessing_results.pkl").exists())
if need_a:
    print("DLinear/NLinear pickles missing -> running Task A (training)…")
    run_script("task_a_dlinear_nlinear.py")
else:
    print("DLinear/NLinear pickles present -> skipping Task A (delete them to retrain).")


## 1. Main text — Tables 3–5, Figure 5, Figure 6 (+ Supplementary Figure S9)

`build_tables_figures.py` writes Tables 3/4/5 (`results/revision2/table{3,4,5}_*.csv/.tex`)
and the comparison panels (b–f) of Figure 5, the full 8-model Figure S9 (b–f), and Figure 6.
`regenerate_panels_a_gj.py` writes Figure 5 panel a (traces) and panels g–j, plus the
matching Figure S9 panels.


In [ ]:
run_script("build_tables_figures.py")
show("figure5_rev2_comparison.png", "figure6_rev2_noise.png", "figureS9_comparison_full8.png")
print("Tables 3/4/5 -> results/revision2/table{3,4,5}_*.csv / .tex")


In [ ]:
run_script("regenerate_panels_a_gj.py")
print("Figure 5 (6-model main):")
show("figure5a_rev2.png", "figure5_panels_gj_rev2.png")
print("Figure S9 (full 8-model):")
show("figureS9a_full8.png", "figureS9_panels_gj_full8.png")


## 2. Supplementary figures (S4, S6, S7, S8, S10)

In [ ]:
run_script("task_b_gamma_sweep.py")
show("figureS_gamma_control_sweep.png")   # Figure S7


In [ ]:
run_script("task_d_realistic_streaming_noise.py")
show("figureS_realistic_streaming_noise.png")   # Figure S8


In [ ]:
run_script("task_f_clean_reanchoring.py")
show("figureS_clean_reanchoring.png")   # Figure S10


In [ ]:
run_script("figure_mode_analysis.py")
show("figureSX_mode_analysis.png")   # Figure S6


In [ ]:
run_script("figure_latent_hparam.py")
show("figureS4.png")   # Figure S4


## 3. Supplementary tables & exports

`task_c_activation.py` produces the control-net activation comparison (Table S4);
`task_e_mode_ablation_export.py` writes the mode-ablation table
(`results/revision2/table_mode_ablation.csv / .tex`).


In [ ]:
run_script("task_c_activation.py")   # Table S4 — control-net activation comparison


In [ ]:
run_script("task_e_mode_ablation_export.py")   # mode-ablation table export


## 4. Original figures (unchanged in revision 2)

The following are **not** modified in this revision and are still generated by the
original notebook `generate_figures_and_tables_revised.ipynb`:

- **Figure 2** — cardiovascular parameter space
- **Figure 3** — Koopman training & latent dynamics
- **Figure 4** — full-horizon prediction (physical + latent)
- **Supplementary S1 / S2 / S3** — Optuna hyperparameter importance
- **Supplementary S5** — all-signal trace comparison
- **Supplementary Table S3** — data-leakage check

Run those cells from the original notebook if you need to regenerate them.


In [ ]:
print("Revision-2 figures currently on disk:\n")
pats = ["figure5a_rev2", "figure5_rev2_comparison", "figure5_panels_gj_rev2",
        "figure6_rev2_noise", "figureS4", "figureSX_mode_analysis",
        "figureS_gamma_control_sweep", "figureS_realistic_streaming_noise",
        "figureS_clean_reanchoring", "figureS9a_full8", "figureS9_comparison_full8",
        "figureS9_panels_gj_full8"]
for stem in pats:
    fp = FIGDIR / f"{stem}.png"
    tag = time.strftime("%Y-%m-%d %H:%M", time.localtime(fp.stat().st_mtime)) if fp.exists() else "MISSING"
    print(f"  {stem:38s}  {tag}")
